In [21]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import torch
from torch import nn
from collections import Counter
from torch.nn.functional import softmax
from torch.nn import CrossEntropyLoss

In [3]:
dataset = load_dataset("go_emotions", "simplified")

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [5]:
label_names = dataset['train'].features["labels"].feature.names
label_names

['admiration',
 'amusement',
 'anger',
 'annoyance',
 'approval',
 'caring',
 'confusion',
 'curiosity',
 'desire',
 'disappointment',
 'disapproval',
 'disgust',
 'embarrassment',
 'excitement',
 'fear',
 'gratitude',
 'grief',
 'joy',
 'love',
 'nervousness',
 'optimism',
 'pride',
 'realization',
 'relief',
 'remorse',
 'sadness',
 'surprise',
 'neutral']

In [6]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: tokenizer(
        x["text"],
        padding="max_length",  
        truncation=True,
        max_length=512
    ),
    batched=True
)

Map: 100%|██████████| 5426/5426 [00:00<00:00, 9883.87 examples/s] 


In [8]:
def simplify_labels(example):
    example["labels"] = example["labels"][0] if len(example["labels"]) > 0 else 27  # class 27 = 'neutral'
    return example

tokenized_dataset = tokenized_dataset.map(simplify_labels)

Map: 100%|██████████| 5426/5426 [00:00<00:00, 15696.39 examples/s]


In [9]:
tokenized_dataset = tokenized_dataset.remove_columns(["text", "id"])
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [20]:
label_counts = Counter([sample["labels"] for sample in tokenized_dataset["train"]])
print(sorted(label_counts.items(), key=lambda x: x[1], reverse=True))

[(tensor(27), 1), (tensor(27), 1), (tensor(2), 1), (tensor(14), 1), (tensor(3), 1), (tensor(26), 1), (tensor(15), 1), (tensor(8), 1), (tensor(0), 1), (tensor(27), 1), (tensor(6), 1), (tensor(1), 1), (tensor(27), 1), (tensor(5), 1), (tensor(3), 1), (tensor(3), 1), (tensor(15), 1), (tensor(2), 1), (tensor(27), 1), (tensor(6), 1), (tensor(6), 1), (tensor(12), 1), (tensor(27), 1), (tensor(27), 1), (tensor(27), 1), (tensor(2), 1), (tensor(27), 1), (tensor(16), 1), (tensor(15), 1), (tensor(27), 1), (tensor(2), 1), (tensor(6), 1), (tensor(27), 1), (tensor(2), 1), (tensor(6), 1), (tensor(17), 1), (tensor(27), 1), (tensor(0), 1), (tensor(25), 1), (tensor(27), 1), (tensor(0), 1), (tensor(15), 1), (tensor(16), 1), (tensor(27), 1), (tensor(7), 1), (tensor(10), 1), (tensor(20), 1), (tensor(27), 1), (tensor(27), 1), (tensor(27), 1), (tensor(27), 1), (tensor(27), 1), (tensor(4), 1), (tensor(27), 1), (tensor(13), 1), (tensor(10), 1), (tensor(27), 1), (tensor(27), 1), (tensor(27), 1), (tensor(15), 1), 

In [10]:
tokenized_dataset["train"].features

{'labels': Value(dtype='int64', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [22]:

label_freqs = Counter([x["labels"] for x in tokenized_dataset["train"]])
num_classes = 28
counts = torch.tensor([label_freqs.get(i, 1) for i in range(num_classes)], dtype=torch.float)
weights = 1.0 / counts
weights = weights / weights.sum()  # normalize

loss_fn = CrossEntropyLoss(weight=weights.to(device))

In [11]:
train_dataset = tokenized_dataset["train"]
val_dataset = tokenized_dataset["validation"]

In [12]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=28).cuda()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)

In [14]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
for epoch in range(6):
    model.train()
    total_loss = 0

    loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")

    for batch in loop:
        batch = {k:v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    print(f"\nEpoch {epoch+1} average loss: {total_loss/len(train_dataloader):.4f}")

Epoch 1: 100%|██████████| 2714/2714 [34:15<00:00,  1.32it/s, loss=0.983]



Epoch 1 average loss: 1.6425


Epoch 2: 100%|██████████| 2714/2714 [34:25<00:00,  1.31it/s, loss=0.448]



Epoch 2 average loss: 1.2499


Epoch 3: 100%|██████████| 2714/2714 [34:25<00:00,  1.31it/s, loss=1.05] 



Epoch 3 average loss: 0.9568


Epoch 4: 100%|██████████| 2714/2714 [34:11<00:00,  1.32it/s, loss=0.0284]



Epoch 4 average loss: 0.6516


Epoch 5:  63%|██████▎   | 1713/2714 [20:49<12:09,  1.37it/s, loss=0.254] 


KeyboardInterrupt: 

In [38]:
torch.save(model.state_dict(), "checkpoint.pt")
torch.save(optimizer.state_dict(), "optimizer.pt")


In [17]:
model.load_state_dict(torch.load("checkpoint.pt"))
optimizer.load_state_dict(torch.load("optimizer.pt"))

In [18]:
model.eval()
correct = 0 
total = 0
with torch.inference_mode():
    for batch in val_dataloader:
        batch = {k:v.to(device) for k,v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)
        labels = batch["labels"]
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    
acc = correct/total
print(f"validation Acc: {acc:.4f}")
        

validation Acc: 0.5428


In [24]:
torch.cuda.is_available()

True